# 02 · Selecionar 200 famílias do PATSTAT com vínculo com o Brasil

**CPU · Sem GPU e sem nova coleta.** Este caderno usa o pacote anterior de 1.000 famílias DOCDB, verifica a sua assinatura e seleciona uma amostra determinística de 200. Se a amostra já estiver instalada, a execução confere o mesmo resultado. Uma saída diferente não é sobrescrita.

O recorte deste piloto considera depósito entre 2010 e 2024 e **qualquer CPC A61 ou C12N** preservado na família. A consulta original incluiu IPC A61, C12N e C12Q. Como o pacote não exportou IPC por família, este filtro usa CPC e essa decisão fica registrada na auditoria. A presença de C12Q não exclui uma família que também tenha A61 ou C12N.

A seleção preserva as antigas partições de treino, validação e teste. Nas janelas 2010–2014, 2015–2019 e 2020–2024, seleciona respectivamente 67, 67 e 66 famílias. O total é 160 para treino, 20 para validação e 20 para teste, com semente 42. A estratificação temporal é balanceada para o piloto e não representa a frequência populacional do PATSTAT.

Antes de começar, copie a pasta atualizada do Desktop ao Drive e execute o notebook 00.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
PASTA_DRIVE = "/content/drive/MyDrive/Replicação LLM EIDT IC"  # @param {type:"string"}
raiz = Path(PASTA_DRIVE).expanduser().resolve(strict=True)
ponto_de_montagem = Path("/content/drive").resolve(strict=True)
if raiz == ponto_de_montagem or not raiz.is_relative_to(ponto_de_montagem) or not raiz.is_dir():
    raise ValueError("Selecione a pasta completa do projeto dentro do Drive montado.")
print("Pasta selecionada:", raiz)

## 1. Conferir programa, configuração e pacote

O arquivo esperado é `dados/brutos/patstat_br_1000/LLM_EIDT_LORA_1000.zip`. Mantenha o ZIP original. O script lê somente os arquivos esperados dentro dele, sem extrair seu conteúdo no Drive. Se o hash divergir, confira a cópia e a documentação de origem antes de continuar.

In [ ]:
import hashlib
import json
import re
import subprocess
import sys

def sha256_arquivo(caminho):
    resumo = hashlib.sha256()
    with caminho.open("rb") as fluxo:
        for bloco in iter(lambda: fluxo.read(1024 * 1024), b""):
            resumo.update(bloco)
    return resumo.hexdigest()

def conferir_integridade(nomes):
    indice = raiz / "SHA256SUMS.txt"
    assinaturas = {}
    for linha in indice.read_text(encoding="utf-8-sig").splitlines():
        correspondencia = re.fullmatch(r"([0-9a-f]{64})  (.+)", linha)
        if correspondencia:
            nome = correspondencia.group(2)
            if nome in assinaturas:
                raise ValueError(f"Hash duplicado no índice para {nome}.")
            assinaturas[nome] = correspondencia.group(1)
    for nome in nomes:
        caminho = raiz / nome
        if nome not in assinaturas or not caminho.is_file() or sha256_arquivo(caminho) != assinaturas[nome]:
            raise ValueError(f"Falha de integridade em {nome}. Recopie o arquivo e o índice da mesma entrega.")
        print("Conferido:", nome)

def caminho_no_projeto(relativo):
    relativo = Path(relativo)
    if relativo.is_absolute() or ".." in relativo.parts:
        raise ValueError("O caminho configurado deve ser relativo à pasta do projeto.")
    caminho = (raiz / relativo).resolve()
    if not caminho.is_relative_to(raiz):
        raise ValueError("O caminho configurado saiu da pasta do projeto.")
    return caminho

conferir_integridade(("config/experimento.json", "scripts/selecionar_piloto_br.py"))
arquivo_config = raiz / "config/experimento.json"
config = json.loads(arquivo_config.read_text(encoding="utf-8-sig"))
if config.get("piloto", {}).get("id") != "br_patstat_200_qwen3_8b_v1":
    raise ValueError("Atualize o projeto para a versão do piloto brasileiro.")
piloto = config["piloto"]
entrada = caminho_no_projeto(piloto["coleta"]["arquivo"])
if not entrada.is_file() or sha256_arquivo(entrada) != piloto["coleta"]["sha256_zip"]:
    raise ValueError("O pacote não foi encontrado ou seu SHA-256 diverge da origem congelada.")
saida_csv = caminho_no_projeto(piloto["amostragem"]["saida"])
saida_auditoria = caminho_no_projeto(piloto["amostragem"]["auditoria"])
if saida_csv.name != "piloto_200.csv" or saida_auditoria != saida_csv.parent / "auditoria.json":
    raise ValueError("Confira os caminhos de saída da seleção na configuração.")
print("Pacote conferido:", entrada.relative_to(raiz))
print("Partições previstas:", piloto["amostragem"]["particoes"])

## 2. Selecionar ou reproduzir a amostra congelada

O programa valida os 1.000 IDs, os rótulos e as partições antigas, aplica o filtro CPC e verifica se há famílias suficientes para cada cota. A ordem de seleção é determinada por SHA-256 com a semente fixada. Nenhuma família muda de partição.

O representante administrativo da família e os textos podem vir de pedidos diferentes. `appln_nr` é número de pedido e não será convertido em identificador de publicação. O rótulo `cpc_proxy` mantém a regra histórica do primeiro código CPC em ordem alfanumérica, com essa limitação explícita.

In [ ]:
subprocess.run(
    [sys.executable, str(raiz / "scripts/selecionar_piloto_br.py"),
     "--entrada", str(entrada), "--saida", str(saida_csv.parent),
     "--config", str(arquivo_config)],
    cwd=raiz,
    check=True,
)
print("Seleção conferida:", saida_csv.relative_to(raiz))

## 3. Conferir contagens, idiomas e auditoria

A auditoria registra o universo recebido, o recorte elegível, as cotas, os idiomas, a cobertura dos rótulos CPC e as assinaturas da entrada e da amostra. Guarde os dois arquivos juntos. Se o programa apontar divergência em uma saída existente, investigue antes de alterar ou substituir o resultado.

In [ ]:
import csv
from collections import Counter

with saida_csv.open(encoding="utf-8-sig", newline="") as arquivo:
    familias = list(csv.DictReader(arquivo))
if len(familias) != 200 or len({item["id_familia_docdb"] for item in familias}) != 200:
    raise RuntimeError("A seleção deve conter exatamente 200 famílias distintas.")
particoes = Counter(item["particao"] for item in familias)
if dict(particoes) != piloto["amostragem"]["particoes"]:
    raise RuntimeError("As partições diferem do protocolo.")
print("Famílias:", len(familias))
print("Partições:", dict(particoes))
print("Janelas:", dict(Counter(item["janela"] for item in familias)))
print("Idioma título/resumo:", dict(Counter(
    item["idioma_titulo"] + "/" + item["idioma_resumo"] for item in familias
)))
auditoria = json.loads(saida_auditoria.read_text(encoding="utf-8"))
if auditoria["sha256_piloto"] != sha256_arquivo(saida_csv):
    raise RuntimeError("A assinatura da amostra difere da auditoria.")
print(json.dumps(auditoria, ensure_ascii=False, indent=2))
print("Auditoria:", saida_auditoria.relative_to(raiz))

## 4. Próximo passo

Execute **03_preprocessar_patstat_br.ipynb** em CPU. Ele preserva os textos originais, prepara a normalização mínima e separa os rótulos CPC das entradas de extração. Volte ao 00 para registrar os novos arquivos se necessário.

Esta amostra é uma adaptação brasileira para testar o pipeline. As frequências, a novidade e as métricas temáticas são exploratórias. Citações, histórico anterior a 2010 e anotações independentes continuam como insumos para avaliações futuras. Não há treinamento ou geração de evidências neste caderno.